<a href="https://colab.research.google.com/github/RajSingh3012/ecommerce-churn-prediction/blob/main/Ecommerce_Analysis_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

customers_path = '/content/olist_customers_dataset.csv'
orders_path = '/content/olist_orders_dataset.csv'
payments_path = '/content/olist_order_payments_dataset.csv'

customers_df = pd.read_csv(customers_path)
orders_df = pd.read_csv(orders_path)
payments_df = pd.read_csv(payments_path)

print("Customers Data:")
print(customers_df.head(2))

print("Orders Data:")
print(orders_df.head(2))

print("\nPayments Data:")
print(payments_df.head(2))

Customers Data:
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
Orders Data:
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1     

In [ ]:
#Merging table based on columns
cust_order_df = pd.merge(customers_df, orders_df, on = 'customer_id', how = 'inner')

print("Shape of merged data:", cust_order_df.shape)
print(cust_order_df.head(2))

master_df = pd.merge(cust_order_df, payments_df, on = 'order_id', how ='inner')
print("Shape of master data:",master_df.shape)
print(master_df.head(2))

Shape of merged data: (99441, 12)
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   

   customer_zip_code_prefix          customer_city customer_state  \
0                     14409                 franca             SP   
1                      9790  sao bernardo do campo             SP   

                           order_id order_status order_purchase_timestamp  \
0  00e7ee1b050b8499577073aeb2a297a1    delivered      2017-05-16 15:05:35   
1  29150127e6685892b6eab3eec79f59c7    delivered      2018-01-12 20:48:24   

     order_approved_at order_delivered_carrier_date  \
0  2017-05-16 15:22:12          2017-05-23 10:47:57   
1  2018-01-12 20:58:32          2018-01-15 17:14:59   

  order_delivered_customer_date order_estimated_delivery_date  
0           2017-05-25 10:35:35           2017-06-05 00:00:00  
1           

In [ ]:
master_df = master_df[['customer_unique_id', 'order_purchase_timestamp', 'payment_value']]

master_df['order_purchase_timestamp'] = pd.to_datetime(master_df['order_purchase_timestamp'])

print(master_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 3 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   customer_unique_id        103886 non-null  object        
 1   order_purchase_timestamp  103886 non-null  datetime64[ns]
 2   payment_value             103886 non-null  float64       
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 2.4+ MB
None


In [ ]:
import datetime as dt
latest_date = master_df['order_purchase_timestamp'].max() + dt.timedelta(days = 1)
rfm_df = master_df.groupby('customer_unique_id').agg({
    'order_purchase_timestamp':lambda x:(latest_date - x.max()).days,
    'customer_unique_id':'count',
    'payment_value':'sum'
})

rfm_df.rename(columns={
    'order_purchase_timestamp': 'Recency',
    'customer_unique_id': 'Frequency',
    'payment_value': 'Monetary'
}, inplace=True)

print(rfm_df.head())

                                  Recency  Frequency  Monetary
customer_unique_id                                            
0000366f3b9a7992bf8c76cfdf3221e2      161          1    141.90
0000b849f77a49e4a4ce2b2a4ca5be3f      164          1     27.19
0000f46a3911fa3c0805444483337064      586          1     86.22
0000f6ccb0745a6a4b88665a16c9f078      370          1     43.62
0004aac84e0df4da2b147fca70cf8255      337          1    196.89


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

scaler = StandardScaler()

rfm_scaled = scaler.fit_transform(rfm_df)

kmeans = KMeans(n_clusters = 4 , random_state = 42)

rfm_df['Customer_Segment'] = kmeans.fit_predict(rfm_scaled)
print(rfm_df.head())

                                  Recency  Frequency  Monetary  \
customer_unique_id                                               
0000366f3b9a7992bf8c76cfdf3221e2      161          1    141.90   
0000b849f77a49e4a4ce2b2a4ca5be3f      164          1     27.19   
0000f46a3911fa3c0805444483337064      586          1     86.22   
0000f6ccb0745a6a4b88665a16c9f078      370          1     43.62   
0004aac84e0df4da2b147fca70cf8255      337          1    196.89   

                                  Customer_Segment  
customer_unique_id                                  
0000366f3b9a7992bf8c76cfdf3221e2                 0  
0000b849f77a49e4a4ce2b2a4ca5be3f                 0  
0000f46a3911fa3c0805444483337064                 1  
0000f6ccb0745a6a4b88665a16c9f078                 1  
0004aac84e0df4da2b147fca70cf8255                 1  


In [ ]:
cluster_summary = rfm_df.groupby('Customer_Segment').mean()
cluster_summary['Customer Segment'] = rfm_df.groupby('Customer_Segment')['Recency'].count()
cluster_summary = cluster_summary.round(2)
print(cluster_summary)

                  Recency  Frequency  Monetary  Customer Segment
Customer_Segment                                                
0                  177.71       1.00    135.31             50506
1                  437.91       1.00    135.20             37587
2                  288.12       1.09   1197.92              2482
3                  289.01       2.37    202.90              5520


In [ ]:
import plotly.express as px
rfm_sample = rfm_df.sample(n=5000, random_state = 42)
rfm_sample['Customer_Segment'] = rfm_sample['Customer_Segment'].astype(str)
fig = px.scatter_3d(
    rfm_sample,
    x = 'Recency',
    y = 'Frequency',
    z = 'Monetary',
    color = 'Customer_Segment',
    title = 'Interactive 3D Customer Segmentation (RFM)',
    opacity = 0.8,
    color_discrete_sequence=px.colors.qualitative.Set1
)
fig.update_layout(margin=dict(l=0, r=0, b=0, t=40))
fig.show()

In [ ]:
final_export_df = rfm_df.reset_index()

final_export_df.to_csv('final_customer_segment.csv', index = False)


In [ ]:
import sqlite3

# 1. Create a connection to a brand new local database file (it will create it automatically)
conn = sqlite3.connect('ecommerce_database.db')

# 2. Write our dataframe directly into a SQL table named 'CustomerSegments'
final_export_df.to_sql('CustomerSegments', conn, if_exists='replace', index=False)

# 3. Always close the connection when you are done!
conn.close()

print("SQL Database Export Successful!")

SQL Database Export Successful!


In [12]:
from sklearn.model_selection import train_test_split

rfm_df['Is_Churned'] = (rfm_df['Recency']>250).astype(int)

rfm_df['Average_Order_Value'] = rfm_df['Monetary']/rfm_df['Frequency']

X = rfm_df.drop(columns = ['Recency','Customer_Segment','Is_Churned'])
y = rfm_df['Is_Churned']

X_train,X_test,y_train,y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

print("Training data shape:",X_train.shape)
print("Testing data shape:",X_test.shape)
print("\nFeatures the model will learn from:")
print(X_train.head(2))

Training data shape: (76876, 3)
Testing data shape: (19219, 3)

Features the model will learn from:
                                  Frequency  Monetary  Average_Order_Value
customer_unique_id                                                        
27859b8fc903e1d19905aa3f011d0904          1     97.76                97.76
abf1d6bd1161f2d55493942abb8c1402          1    112.10               112.10


In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

rf_model = RandomForestClassifier(n_estimators = 100, random_state =42)

print("Training the Random Forest...")
rf_model.fit(X_train, y_train)

predictions = rf_model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Detailed Classification Report:")
print(classification_report(y_test, predictions))



Training the Random Forest...
Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.70      0.65      0.67      8842
           1       0.72      0.76      0.74     10377

    accuracy                           0.71     19219
   macro avg       0.71      0.70      0.70     19219
weighted avg       0.71      0.71      0.71     19219



In [14]:
import pandas as pd

importances = pd.Series(rf_model.feature_importances_, index = X.columns)

print("Feature Importance for Predicting Churn:")
print(importances.sort_values(ascending = False))

Feature Importance for Predicting Churn:
Monetary               0.499926
Average_Order_Value    0.497972
Frequency              0.002102
dtype: float64


In [16]:
from sklearn.model_selection import cross_val_score

print("Running 5-Fold Cross Validation.")
cv_scores = cross_val_score(rf_model, X, y, cv = 5)
print(f"\nScores for the 5 individual exams:{cv_scores}")

true_average = cv_scores.mean()
print(f"True Average Accuracy:{true_average *100:2f}%")

Running 5-Fold Cross Validation.

Scores for the 5 individual exams:[0.70617618 0.7112753  0.70534367 0.71023466 0.70789323]
True Average Accuracy:70.818461%
